# 01 — Exploratory Data Analysis Dataset Klasifikasi Sampah
Memahami struktur dataset, distribusi kelas, karakteristik visual, kondisi berkas, duplikasi persis, dan sumber gambar sebelum eksperimen EfficientNet-B0.

Notebook hanya membaca data: tidak mengubah gambar/label, membuat split, mengunduh bobot, melatih model, atau mengakses layanan eksternal. Semua gambar diproses satu per satu di memori.

**Cara pakai:** simpan di `/home/sembiring/machine_learning/00_pr0j3ct/kompetisi-sampah/notebooks/01_eda_dataset.ipynb`, pilih kernel Python di VS Code/Jupyter, lalu **Run All**. Pastikan folder `train` sudah diekstrak. Jika lokasi berbeda, ubah `TRAIN_DIR` di bagian 2 menjadi path absolut.

Jika paket belum tersedia, pasang secara manual di terminal lingkungan Python yang sama: `python -m pip install pandas matplotlib Pillow ipykernel`. Notebook tidak menjalankan instalasi otomatis. Output diperbarui ketika dijalankan pada data lokal Anda; jangan menyamakan arsip lampiran dengan seluruh dataset kompetisi.


## 2 — Import dan konfigurasi
`pathlib` menangani path, `random` membuat sampel konsisten, `hashlib` memeriksa isi biner, pandas menyusun tabel, matplotlib menggambar grafik, dan Pillow membaca gambar. Tidak diperlukan GPU. `display` adalah fasilitas tampilan tabel Jupyter.


In [ ]:
from pathlib import Path
import random
import hashlib
try:
    import pandas as pd
    import matplotlib.pyplot as plt
    from PIL import Image, ImageOps
except ImportError as exc:
    raise ImportError("Paket belum tersedia. Jalankan manual di terminal: python -m pip install pandas matplotlib Pillow ipykernel; lalu pilih kernel yang sama.") from exc

try:
    display
except NameError:
    display = print

TRAIN_DIR = Path("/home/sembiring/machine_learning/00_pr0j3ct/kompetisi-sampah/train")
PROJECT_DIR = TRAIN_DIR.parent
SOURCES_FILE = PROJECT_DIR / "train_sources.csv"
SEED = 42
EXPECTED_CLASSES = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]
CLASS_TO_LABEL = {name: i for i, name in enumerate(EXPECTED_CLASSES)}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff", ".gif", ".jfif"}
pd.set_option("display.max_colwidth", 100)
plt.rcParams.update({"figure.dpi": 110})
print("Folder yang diperiksa:", TRAIN_DIR)


## 3 — Struktur folder dan inventaris
Pemindaian rekursif dilakukan sekali pada folder kelas yang diharapkan. Folder tambahan dan berkas di root dilaporkan terpisah, bukan otomatis diberi label kompetisi. Ekstensi hanya menentukan kandidat gambar, bukan menjamin berkas dapat dibaca. Symlink dilewati agar satu target tidak dihitung berulang.


In [ ]:
if not TRAIN_DIR.is_absolute():
    raise ValueError("Gunakan path absolut untuk TRAIN_DIR.")
train_exists = TRAIN_DIR.is_dir()
folders = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir()) if train_exists else []
missing_classes = sorted(set(EXPECTED_CLASSES) - set(folders))
extra_folders = sorted(set(folders) - set(EXPECTED_CLASSES))
print("Folder ditemukan:", folders)
print("Kelas hilang:", missing_classes)
print("Folder tambahan (tidak dihitung):", extra_folders)
if not train_exists:
    print(f"PERINGATAN: {TRAIN_DIR} tidak ditemukan. Ekstrak train.zip lalu ubah TRAIN_DIR ke folder train yang benar. Hasil kosong berikut bukan bukti dataset kosong.")

records, non_images, skipped_links = [], [], []
for class_name in EXPECTED_CLASSES:
    class_dir = TRAIN_DIR / class_name
    if class_dir.is_symlink():
        skipped_links.append(class_name)
        continue
    if not class_dir.is_dir():
        continue
    for path in sorted(class_dir.rglob("*")):
        relative = path.relative_to(TRAIN_DIR).as_posix()
        if path.is_symlink():
            skipped_links.append(relative)
        elif path.is_file():
            if path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append({"path": path, "relative_path": relative, "class_name": class_name,
                                "label": CLASS_TO_LABEL[class_name], "extension": path.suffix.lower()})
            else:
                non_images.append(relative)
files_df = pd.DataFrame(records, columns=["path", "relative_path", "class_name", "label", "extension"])
root_files = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_file()) if train_exists else []
print("Berkas root (di luar kelas):", root_files)
print("Kandidat gambar:", len(files_df), "| Non-gambar dalam kelas:", len(non_images))
print("Non-gambar:", non_images)
print("Symlink dilewati:", skipped_links)
nested_df = files_df[files_df["relative_path"].map(lambda p: p.count("/") > 1).astype(bool)]
print("Kandidat dalam subfolder:", len(nested_df))
display(nested_df[["relative_path", "class_name"]])


## 4 — Distribusi kelas
Penyebut persentase adalah seluruh kandidat berkas gambar dalam lima kelas. Jumlah yang lolos pemeriksaan integritas baru dilaporkan pada bagian 7.


In [ ]:
counts = files_df["class_name"].value_counts().reindex(EXPECTED_CLASSES, fill_value=0)
distribution = counts.rename_axis("class_name").reset_index(name="candidate_count")
distribution["percent"] = (distribution["candidate_count"] / len(files_df) * 100).round(2) if len(files_df) else 0.0
display(distribution)
print("Total kandidat:", len(files_df))
if len(files_df):
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(distribution["class_name"], distribution["candidate_count"], color="#377c95")
    ax.bar_label(bars, padding=3)
    ax.set(title="Distribusi kandidat berkas gambar", xlabel="Kelas folder", ylabel="Jumlah berkas")
    ax.set_ylim(0, counts.max() * 1.18)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout(); plt.show()
    print("Terbanyak:", counts[counts == counts.max()].index.tolist(), "=", counts.max())
    print("Tersedikit:", counts[counts == counts.min()].index.tolist(), "=", counts.min())
    print("Perbedaan jumlah kelas belum cukup untuk memilih oversampling atau class weighting. Evaluasi Macro-F1 dan F1 tiap kelas diperlukan pada tahap berikutnya.")
else:
    print("Distribusi belum dapat diinterpretasikan: tidak ada kandidat yang ditemukan.")


## 5 — Sampel visual per kelas
Enam sampel dipilih tanpa penggantian dengan seed 42 dari daftar path terurut. Label berasal dari folder, bukan prediksi. Orientasi EXIF diterapkan hanya pada tampilan; rasio gambar dipertahankan. Bila satu sampel gagal dibaca, sampel lain tetap ditampilkan. Sampel ini bukan audit seluruh label.


In [ ]:
rng = random.Random(SEED)
sample_paths = []
for class_name in EXPECTED_CLASSES:
    candidates = files_df.loc[files_df["class_name"] == class_name, "path"].tolist()
    selected = rng.sample(candidates, min(6, len(candidates)))
    sample_paths.extend(p.relative_to(TRAIN_DIR).as_posix() for p in selected)
    if not selected:
        print(class_name, ": tidak ada sampel.")
        continue
    fig, axes = plt.subplots(2, 3, figsize=(12, 8), squeeze=False)
    fig.suptitle(f"{class_name} — label {CLASS_TO_LABEL[class_name]}")
    for ax in axes.flat:
        ax.axis("off")
    for ax, path in zip(axes.flat, selected):
        try:
            with Image.open(path) as im:
                shown = ImageOps.exif_transpose(im).convert("RGB")
                ax.imshow(shown)
            ax.set_title(path.name, fontsize=8, wrap=True)
        except Exception as exc:
            ax.text(0.05, 0.5, "Gagal dibaca", transform=ax.transAxes)
            print("Gagal menampilkan", path.relative_to(TRAIN_DIR), type(exc).__name__, str(exc))
    plt.tight_layout(); plt.show()
print("Path sampel untuk pencatatan manual:")
for relative in sample_paths:
    print(relative)


**Panduan interpretasi visual:** periksa bentuk/material, ukuran objek dalam bingkai, sudut, kondisi objek, latar/pencahayaan, tangan manusia, tulisan/logo/watermark, dan kemiripan antarkelas. Botol memiliki leher/tutup ulir; cup plastik tidak memiliki leher. Gelas kertas masuk `kertas_kardus`, sedangkan gelas gabus masuk `styrofoam`. Catat hanya ciri yang benar-benar terlihat di bagian 9. Tulisan promosi tidak otomatis berarti label salah.


## 6 — Metadata seluruh kandidat
Satu pemeriksaan per berkas menggabungkan `verify()` lalu membuka ulang dan menjalankan `load()` untuk memastikan dekode piksel pertama berhasil. Hasil disimpan agar bagian integritas tidak perlu membaca ulang. Dimensi di tabel adalah dimensi asli berkas; orientasi EXIF 5–8 menukar sumbu saat ditampilkan. Untuk gambar multiframe, pemeriksaan decode ini hanya pada frame pertama.


In [ ]:
def inspect_image(path):
    result = {"format": None, "width": None, "height": None, "aspect_ratio": None,
              "mode": None, "size_bytes": None, "exif_orientation": None,
              "frame_count": None, "read_ok": False, "error": ""}
    try:
        result["size_bytes"] = path.stat().st_size
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            im.load()
            width, height = im.size
            result.update(format=im.format, width=width, height=height,
                          aspect_ratio=width / height, mode=im.mode,
                          exif_orientation=im.getexif().get(274, 1),
                          frame_count=getattr(im, "n_frames", 1), read_ok=True)
    except Exception as exc:
        result["error"] = f"{type(exc).__name__}: {exc}"
    return result

metadata_columns = ["format", "width", "height", "aspect_ratio", "mode", "size_bytes", "exif_orientation", "frame_count", "read_ok", "error"]
metadata_df = pd.DataFrame([inspect_image(path) for path in files_df["path"]], columns=metadata_columns)
images_df = pd.concat([files_df, metadata_df], axis=1)
valid_df = images_df[images_df["read_ok"].eq(True)].copy()
display(images_df.drop(columns="path").head(10))
if len(valid_df):
    display(valid_df[["width", "height", "aspect_ratio", "size_bytes"]].describe())
    for column in ["extension", "format", "mode", "exif_orientation", "frame_count"]:
        display(valid_df[column].value_counts(dropna=False).rename_axis(column).reset_index(name="count"))
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, column, label in zip(axes, ["width", "height", "aspect_ratio"], ["Lebar asli (px)", "Tinggi asli (px)", "Rasio lebar/tinggi"]):
        ax.hist(valid_df[column], bins=20, color="#377c95", edgecolor="white")
        ax.set(xlabel=label, ylabel="Jumlah", title=label)
    plt.tight_layout(); plt.show()
    print("Variasi mode/dimensi adalah karakteristik data, bukan alasan otomatis menghapus gambar. Pertimbangkan konversi RGB dan transformasi ukuran di memori pada tahap training.")
else:
    print("Belum ada metadata gambar yang lolos pemeriksaan.")


## 7 — Integritas gambar
Ringkasan menggunakan hasil bagian 6. Berkas yang gagal tetap berada di lokasi asal. Kelulusan pemeriksaan pembacaan bukan bukti kebenaran label atau kualitas semantik gambar.


In [ ]:
failed_df = images_df[~images_df["read_ok"].eq(True)]
print("Kandidat:", len(images_df))
print("Lolos verify + decode frame pertama:", len(valid_df))
print("Gagal diperiksa:", len(failed_df))
display(failed_df[["relative_path", "error"]])
read_counts = valid_df["class_name"].value_counts().reindex(EXPECTED_CLASSES, fill_value=0)
integrity_by_class = distribution[["class_name", "candidate_count"]].copy()
integrity_by_class["readable_count"] = integrity_by_class["class_name"].map(read_counts)
integrity_by_class["failed_count"] = integrity_by_class["candidate_count"] - integrity_by_class["readable_count"]
display(integrity_by_class)
if len(images_df) and not len(failed_df):
    print("Tidak ditemukan masalah pembacaan berdasarkan pemeriksaan verify dan decode frame pertama yang dilakukan.")


## 8 — Duplikat persis dengan SHA-256
Hash dihitung sekali per berkas yang lolos pemeriksaan, dalam potongan 1 MiB. SHA-256 membandingkan isi biner: gambar yang di-resize, di-encode ulang, atau diberi watermark dapat terlihat sama tetapi memiliki hash berbeda. Konflik lintas kelas hanya kandidat untuk ditinjau.


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

hash_rows, hash_errors = [], []
for row in valid_df.itertuples(index=False):
    try:
        hash_rows.append({"relative_path": row.relative_path, "class_name": row.class_name, "sha256": sha256_file(row.path)})
    except OSError as exc:
        hash_errors.append({"relative_path": row.relative_path, "error": str(exc)})
hashes_df = pd.DataFrame(hash_rows, columns=["relative_path", "class_name", "sha256"])
duplicates_df = hashes_df[hashes_df.duplicated("sha256", keep=False)].copy()
duplicate_groups = []
for number, (digest, group) in enumerate(duplicates_df.groupby("sha256", sort=True), start=1):
    cross_class = group["class_name"].nunique() > 1
    duplicate_groups.append({"group": number, "sha256": digest, "file_count": len(group), "cross_class": cross_class})
    print(f"Kelompok {number} | lintas kelas: {cross_class} | SHA-256: {digest}")
    display(group[["relative_path", "class_name"]])
print("Hash berhasil:", len(hashes_df), "| Gagal hash:", len(hash_errors))
print("Kelompok duplikat persis:", len(duplicate_groups))
if hash_errors:
    display(pd.DataFrame(hash_errors))
if not len(duplicates_df):
    print("Tidak ditemukan duplikat biner pada berkas yang berhasil dihitung hash-nya; kemiripan visual belum diuji.")


## 9 — Pemeriksaan awal potensi label noise
- **Label noise:** objek kemungkinan tidak sesuai label folder dan definisi kelas.
- **Watermark/poster:** ciri visual yang berpotensi menjadi jalan pintas model; bukan otomatis salah label.
- **Ambigu:** material/objek tidak cukup jelas, atau beberapa jenis sampah muncul bersama.
- **Kesalahan prediksi:** belum bisa dianalisis karena tidak ada model yang dilatih.

Isi catatan di bawah menggunakan path persis dari bagian 5. Tidak ada kolom label pengganti dan tidak ada perubahan dataset. Catatan kosong berarti belum ada kandidat yang dicatat, bukan berarti seluruh label benar. Setelah baseline di notebook 02, prediksi salah/loss tinggi hanya menjadi kandidat pemeriksaan ulang.


In [ ]:
# Tambahkan kamus sesuai hasil pengamatan sendiri:
# {"relative_path": "kelas/nama_file.jpg", "category": "ambigu", "reason": "Alasan visual yang terlihat"}
manual_review = []
visual_observations = []  # Isi deskripsi singkat ciri yang benar-benar terlihat.
review_df = pd.DataFrame(manual_review, columns=["relative_path", "category", "reason"])
if len(review_df):
    review_df["path_found"] = review_df["relative_path"].isin(files_df["relative_path"])
    display(review_df)
    if not review_df["path_found"].all():
        print("Ada path catatan yang tidak cocok dengan inventaris; periksa kembali penulisannya.")
else:
    print("Belum ada kandidat manual yang dicatat dalam sesi ini.")
for observation in visual_observations:
    print("-", observation)


## 10 — Pemeriksaan train_sources.csv
CSV bersifat opsional. Periksa struktur aktual sebelum memilih kolom. Sel pertama tidak mengasumsikan nama kolom dan tidak melakukan join. Ringkasan nilai ditampilkan hanya untuk kolom kategorikal berjumlah nilai unik rendah; kolom lain tetap dilaporkan jumlah uniknya.


In [ ]:
sources_df = None
source_status = "Tidak tersedia"
if not SOURCES_FILE.is_file():
    print(f"PERINGATAN: {SOURCES_FILE} tidak tersedia. Analisis sumber dilewati; EDA gambar tetap berjalan.")
else:
    try:
        sources_df = pd.read_csv(SOURCES_FILE)
        source_status = "Berhasil dibaca"
        print("Ukuran dataframe:", sources_df.shape)
        print("Kolom aktual:", sources_df.columns.tolist())
        display(sources_df.head())
        display(pd.DataFrame({"dtype": sources_df.dtypes.astype(str), "missing": sources_df.isna().sum(), "unique": sources_df.nunique(dropna=True)}))
        print("Baris duplikat identik (di luar kemunculan pertama):", sources_df.duplicated().sum())
        for column in sources_df.columns:
            if 0 < sources_df[column].nunique() <= 30:
                display(sources_df[column].value_counts(dropna=False).head(15).rename_axis(str(column)).reset_index(name="count"))
    except Exception as exc:
        source_status = f"Gagal dibaca: {type(exc).__name__}: {exc}"
        print(source_status, "Periksa format, delimiter, atau encoding CSV; bagian lain tetap dapat berjalan.")


**Pemetaan opsional setelah struktur diperiksa.** Pilih kolom secara eksplisit di bawah hanya jika artinya telah dipastikan. Aturan join yang didukung adalah path relatif lengkap terhadap `train`, misalnya `botol_plastik/foto.jpg`; pencocokan persis tanpa menebak berdasarkan basename. Kolom sumber dipilih berdasarkan arti sebenarnya. Bila CSV menggunakan format lain, biarkan kosong sampai aturan relasinya dapat dipastikan. Duplikat kunci diblokir agar tidak menggandakan jumlah gambar.


In [ ]:
CSV_PATH_COLUMN = None
CSV_SOURCE_COLUMN = None
source_distribution = None
if sources_df is not None:
    if CSV_PATH_COLUMN is None or CSV_SOURCE_COLUMN is None:
        print("Analisis sumber-kelas belum dilakukan: kolom dan relasi belum dikonfirmasi.")
    elif CSV_PATH_COLUMN == CSV_SOURCE_COLUMN or not {CSV_PATH_COLUMN, CSV_SOURCE_COLUMN}.issubset(sources_df.columns):
        print("Pilih dua kolom berbeda yang benar-benar tersedia dan sesuai maknanya.")
    else:
        mapping = sources_df[[CSV_PATH_COLUMN, CSV_SOURCE_COLUMN]].copy()
        mapping.columns = ["relative_path", "source_value"]
        if mapping["relative_path"].isna().any() or mapping["relative_path"].duplicated().any():
            print("Join dibatalkan: kunci file kosong/berulang. Tinjau CSV secara manual tanpa mengubah data asli.")
        else:
            matched = files_df[["relative_path", "class_name"]].merge(mapping, on="relative_path", how="left", validate="one_to_one", indicator=True)
            print("Aturan: path relatif persis terhadap train.")
            print("Kolom path:", CSV_PATH_COLUMN, "| Kolom sumber:", CSV_SOURCE_COLUMN)
            print("Gambar cocok:", matched["_merge"].eq("both").sum(), "/", len(files_df))
            print("Baris CSV tidak cocok:", (~mapping["relative_path"].isin(files_df["relative_path"])).sum())
            source_distribution = matched.loc[matched["_merge"].eq("both")].groupby(["class_name", "source_value"], dropna=False).size().reset_index(name="image_count")
            display(source_distribution)
            print("Konsentrasi sumber hanya menggambarkan data yang cocok; bukan bukti model pasti bias.")


## 11 — Ringkasan dan rekomendasi
Pisahkan angka terverifikasi, kandidat yang perlu ditinjau, dan rencana eksperimen berikutnya. Ringkasan ini hanya berlaku pada folder yang benar-benar dipindai.


In [ ]:
summary = {
    "Folder training tersedia": train_exists,
    "Kelas yang diharapkan ditemukan": len(set(folders) & set(EXPECTED_CLASSES)),
    "Folder tambahan": len(extra_folders),
    "Kandidat gambar": len(files_df),
    "Non-gambar dalam kelas": len(non_images),
    "Lolos pemeriksaan pembacaan": len(valid_df),
    "Gagal pemeriksaan pembacaan": len(failed_df),
    "Kelompok duplikat biner": len(duplicate_groups),
    "Kelompok duplikat lintas kelas": sum(g["cross_class"] for g in duplicate_groups),
    "Gagal hash": len(hash_errors),
    "Status CSV sumber": source_status,
    "Kandidat dicatat manual": len(review_df),
}
display(pd.DataFrame(summary.items(), columns=["Pemeriksaan", "Hasil"]))
display(distribution)
if sources_df is not None:
    print("Struktur CSV:", sources_df.shape, sources_df.columns.tolist())
print("Kandidat untuk ditinjau: berkas gagal dibaca, kelompok hash lintas kelas, dan catatan manual (jika ada). Tidak ada tindakan koreksi otomatis.")
if visual_observations:
    print("Pengamatan visual yang dicatat:")
    for observation in visual_observations:
        print("-", observation)
else:
    print("Belum ada pengamatan visual tertulis dalam variabel visual_observations. Lihat sampel dan catatan pemeriksaan lampiran di akhir notebook.")


### Persiapan untuk 02_efficientnet_b0.ipynb
1. Pastikan folder data lengkap dan tinjau kegagalan baca/konflik label secara manual sebelum eksperimen.
2. Rencanakan split lokal yang tetap dengan seed tercatat dan representasi tiap kelas. Bila terdapat duplikat atau kelompok sumber yang relevan, cegah gambar terkait tersebar antara train dan validation. Split belum dibuat di sini.
3. Rencanakan konversi RGB, penanganan orientasi EXIF, ukuran input dan normalisasi sesuai pretrained weights ImageNet yang akan dipilih. Jangan menyimpan ulang seluruh gambar asli. Pertimbangkan efek crop pada objek dan rasio aspek yang beragam.
4. Gunakan augmentasi hanya pada training dengan kekuatan yang diuji; validation menggunakan transformasi deterministik.
5. Mulai baseline EfficientNet-B0 pretrained ImageNet, lalu ukur Macro-F1, F1 per kelas, dan confusion matrix pada split lokal yang konsisten. Angka hidden validation dosen bukan hasil eksperimen notebook ini dan tidak langsung sebanding dengan split lokal.
6. Tentukan kebutuhan class weighting atau sampling setelah melihat hasil evaluasi, bukan hanya selisih jumlah kelas. Tinjau prediksi salah/loss tinggi secara manual; jangan otomatis mengganti label.

Notebook ini tidak menyatakan dataset bebas label noise dan tidak menjanjikan skor tertentu.


### Catatan pemeriksaan aktual pada lampiran train.zip — 14 Oktober 2026
Sel kode telah diuji berurutan pada salinan hasil ekstraksi ZIP di lingkungan pengerjaan, menggunakan penyesuaian path sementara. Konfigurasi notebook yang dikirim tetap menggunakan path Ubuntu yang diminta. Output sel dikosongkan agar hasil lampiran tidak disalahartikan sebagai hasil folder lokal lengkap, dan agar berkas notebook tidak menyertakan salinan gambar. Jalankan **Run All** untuk menghasilkan tabel dan visualisasi Anda sendiri.

**Temuan terverifikasi pada lampiran:**

| Kelas | Kandidat | Persentase |
|---|---:|---:|
| botol_plastik | 45 | 18,91% |
| gelas_plastik | 45 | 18,91% |
| kertas_kardus | 32 | 13,45% |
| saset_kemasan | 58 | 24,37% |
| styrofoam | 58 | 24,37% |

Total **238** kandidat; kelima folder ada; tidak ditemukan folder tambahan maupun berkas non-gambar di dalam kelas. Semua 238 lolos `verify()` dan decode frame pertama; tidak ada kegagalan hash dan tidak ditemukan kelompok duplikat biner. Seluruh gambar yang diperiksa bermode RGB. Lebar asli 160–1024 piksel, tinggi asli 154–1024 piksel. `train_sources.csv` tidak tersedia. Angka ini hanya berlaku untuk ZIP yang dikirim, bukan keseluruhan sekitar 2.900 gambar dalam deskripsi tugas. Jumlah persentase dapat berbeda 0,01% karena pembulatan.

**Pengamatan pada 30 sampel seed 42:** botol ditampilkan sebagai produk tunggal, dua botol di meja, dan tumpukan sampah. Sampel gelas plastik memuat foto cup, minuman, sedotan, dan poster produk. Sampel kertas/kardus memuat bundel kertas, kertas terbuang, dan poster paper cup; gelas kertas sesuai definisi kelas ini. Sampel saset memuat bungkus makanan/kopi, tumpukan kemasan, dan gambar kerajinan bungkus snack. Sampel styrofoam memuat kotak, potongan berbentuk lingkaran, dan objek di antara sampah lain. Latar polos maupun lingkungan nyata, teks merek/promosi, dan watermark terlihat. Pengamatan tersebut tidak mewakili pemeriksaan seluruh label.

**Kandidat tinjauan manual, bukan label salah yang terkonfirmasi:**

| Path relatif | Alasan pemeriksaan |
|---|---|
| `gelas_plastik/gelas_plastik_db57079d67f87e8d.jpg` | Poster bertuliskan Plastik Sealer Cup lebih menonjolkan gulungan film penutup; perlu ditinjau apakah objek memenuhi definisi cup plastik atau merupakan gambar produk terkait. |
| `saset_kemasan/saset_kemasan_899ec1b154eb78fc.jpg` | Kemasan berdiri terlihat menyerupai pouch, tetapi material kertas/plastik tidak dapat dipastikan dari sampel; tinjau gambar penuh sesuai definisi kelas. |

Tidak ada label pengganti yang ditetapkan. Kedua catatan ini berasal dari lampiran dan perlu diverifikasi keberadaan serta tampilannya bila memakai dataset lain. Untuk mencatatnya pada ringkasan dinamis sesi lokal, salin path dan alasan ke `manual_review` setelah memeriksa sampel Anda.

**Implikasi:** pastikan data lokal lengkap sebelum notebook 02. Variasi resolusi mendukung persiapan transformasi ukuran di memori, sedangkan teks promosi/latar menjadi aspek yang perlu diperhatikan saat evaluasi. Tidak ada bukti dari EDA ini bahwa model pasti bias atau bahwa seluruh label sudah benar. Rencana split, augmentasi, pretrained ImageNet, dan evaluasi Macro-F1 tetap merupakan pekerjaan tahap berikutnya.
